In [1]:
%cd ProyectoFinal

/home/jovyan/work/ProyectoFinal


In [2]:
import pandas as pd
import requests
import json
from datetime import datetime
import os
import re
import subprocess
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import shutil
import holidays
from datetime import date

# API Demanda Electrica ESIOS


In [4]:
def obtener_demanda_ree(fecha_inicio, fecha_fin, token):
    # ID 1293: Demanda real
    id_indicador = "1293"
    url = f"https://api.esios.ree.es/indicators/{id_indicador}"
    
    headers = {
        'Accept': 'application/json; application/vnd.esios-api.v1+json',
        'Content-Type': 'application/json',
        'Authorization': f'Token token="{token}"'
    }
    
    params = {
        'start_date': f'{fecha_inicio}T00:00',
        'end_date': f'{fecha_fin}T23:59',
        'time_trunc': 'hour'  # Forzamos granularidad horaria
    }
    
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        # Extraer los valores del JSON
        valores = data['indicator']['values']
        
        # Crear el DataFrame
        df = pd.DataFrame(valores)
        
        # Convertir columna de fecha a datetime y formatear
        df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
        
        # Seleccionar y renombrar columnas relevantes
        df = df[['datetime', 'value']]
        df.columns = ['fecha_hora', 'demanda_mw']
        
        return df
    else:
        print(f"Error: {response.status_code}")
        return None

# --- CONFIGURACIÓN ---
# Solicita tu token en: consultasios@ree.es
MI_TOKEN = "TU_TOKEN_AQUÍ" 
inicio = "2024-03-01"
fin = "2024-03-05"

df_demanda = obtener_demanda_ree(inicio, fin, MI_TOKEN)

if df_demanda is not None:
    print(df_demanda.head())

Error: 403


# Api Datos

- El IRE es mensual, siempre debes usar month
- Cuando intent buscar la demanda solo es posible a ver la granularidad en meses, no me deja poner laopcion de dias o da error

In [23]:
import requests
import pandas as pd

def obtener_balance_renovable(fecha_inicio, fecha_fin):
    url = "https://apidatos.ree.es/es/datos/balance/balance-electrico"
    
    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/json',
        'Host': 'apidatos.ree.es'
    }
    
    params = {
        'start_date': f'{fecha_inicio}T00:00',
        'end_date': f'{fecha_fin}T23:59',
        'time_trunc': 'day'
    }
    
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        lista_dfs = []
        
        for item in data.get('included', []):
            # Filtramos estrictamente por el bloque de "Renovable"
            if item.get('type') == "Renovable":
                
                # Recorremos cada tipo de energía (Eólica, Hidráulica, etc.)
                for sub_item in item.get('attributes', {}).get('content', []):
                    tipo_nombre = sub_item.get('type')

                    # Nos quedamos solo con un tipo de energia.
                    if tipo_nombre == "Eólica":

                        valores = sub_item.get('attributes', {}).get('values', [])
                        
                        if valores:
                            temp_df = pd.DataFrame(valores)
                            temp_df['tipo_energia'] = tipo_nombre
                            lista_dfs.append(temp_df)
        
        if not lista_dfs:
            print("No se encontraron datos renovables.")
            return None
            
       # Combinamos todos los datos
        df_final = pd.concat(lista_dfs, ignore_index=True)
        
        # --- CAMBIO AQUÍ: Conversión robusta ---
        # 1. Convertimos a datetime asegurando que detecte el formato ISO de la API
        df_final['datetime'] = pd.to_datetime(df_final['datetime'], utc=True)
        
        # 2. Ahora que es "datetimelike", quitamos la zona horaria (hacemos el 'naive')
        df_final['datetime'] = df_final['datetime'].dt.tz_localize(None)
        # ---------------------------------------
        
        # Seleccionamos las columnas solicitadas
        # Asegúrate de que 'value' esté en la lista si lo añadiste manualmente
        columnas_disponibles = ['tipo_energia', 'datetime', 'percentage', 'value']
            
        df_final = df_final[columnas_disponibles]
        
        # Renombramos columnas
        df_final.columns = ['Tipo Energía', 'Fecha', 'Porcentaje (%)', 'Valor']
        
        # Multiplicamos por 100 para tener formato porcentaje (0.31 -> 31.0)
        df_final['Porcentaje (%)'] = (df_final['Porcentaje (%)'] * 100).round(2)
        
        return df_final
    else:
        print(f"Error en la API: {response.status_code}")
        return None

# --- EJECUCIÓN ---
# Nota: La API de REE a veces tiene límites de rango, un mes suele estar bien.
inicio = "2026-01-01"
fin = date.today().strftime("%Y-%m-%d")

df_resultado = obtener_balance_renovable(inicio, fin)

if df_resultado is not None:
    # Ordenamos cronológicamente para ver la evolución
    df_resultado = df_resultado.sort_values(by=['Fecha', 'Porcentaje (%)'], ascending=[True, False])
    print(df_resultado.to_string(index=False))

Tipo Energía               Fecha  Porcentaje (%)      Valor
      Eólica 2025-12-31 23:00:00           45.23 103103.387
      Eólica 2026-01-01 23:00:00           46.45 134837.404
      Eólica 2026-01-02 23:00:00           49.32 142619.826
      Eólica 2026-01-03 23:00:00           76.50 356417.339
      Eólica 2026-01-04 23:00:00           62.46 280672.555
      Eólica 2026-01-05 23:00:00           60.47 296005.664
      Eólica 2026-01-06 23:00:00           56.62 281092.073
      Eólica 2026-01-07 23:00:00           66.74 373141.383
      Eólica 2026-01-08 23:00:00           75.50 456779.582
      Eólica 2026-01-09 23:00:00           59.58 296453.285
      Eólica 2026-01-10 23:00:00           52.01 229100.064
      Eólica 2026-01-11 23:00:00           49.05 211120.058
      Eólica 2026-01-12 23:00:00           57.91 264705.430
      Eólica 2026-01-13 23:00:00           27.09  87505.912
      Eólica 2026-01-14 23:00:00           52.34 217920.575
      Eólica 2026-01-15 23:00:00        

# AEMET HTML

In [ ]:
!conda install -c conda-forge unrar -y

In [12]:

# --- CONSTANTES ---
CONFIG = {
    "URL_WEB": "https://datosclima.es/Aemet2013/DescargaDatos.html",
    "BASE_URL": "https://datosclima.es/Aemet2013/",
    "TEMP_DIR": os.path.abspath("data/datos_clima_aemet"),
    "ARCHIVO_FINAL": os.path.abspath("data/datos_clima_aemet/datos_clima_2014_2026.csv"),
    "ULTIMA_FECHA_REGISTRADA": os.path.abspath("data/datos_clima_aemet/ultima_fecha_registrada.csv"),
    "HEADERS": {'User-Agent': 'Mozilla/5.0'}
}

def inicializar_entorno():
    """Crea directorios necesarios si no existen."""
    os.makedirs(CONFIG["TEMP_DIR"], exist_ok=True)

def obtener_ultima_fecha_registrada():
    """Detecta la última fecha en el CSV sin cargar todo el archivo en memoria."""
    if not os.path.exists(CONFIG["ARCHIVO_FINAL"]):
        return 0
    if not os.path.exists(CONFIG["ULTIMA_FECHA_REGISTRADA"]):
        try:
            # Leemos solo la última fila para optimizar memoria
            df_last = pd.read_csv(CONFIG["ARCHIVO_FINAL"], skipinitialspace=True).tail(1)
            if df_last.empty or 'fecha' not in df_last.columns:
                return 0
                
            fecha_dt = pd.to_datetime(df_last['fecha'].iloc[0], errors='coerce')
            if pd.notnull(fecha_dt):
                print(f"📅 Último registro local: {fecha_dt.date()}")
                return int(fecha_dt.strftime('%Y%m'))
        except Exception as e:
            print(f"⚠️ Error leyendo histórico: {e}")
        return 0
    
    try:
        # Leemos solo la última fila para optimizar memoria
        df_last = pd.read_csv(CONFIG["ULTIMA_FECHA_REGISTRADA"], skipinitialspace=True).tail(1)
        if df_last.empty or 'fecha' not in df_last.columns:
            return 0
            
        fecha_dt = pd.to_datetime(df_last['fecha'].iloc[0], errors='coerce')
        if pd.notnull(fecha_dt):
            print(f"📅 Último registro local: {fecha_dt.date()}")
            return int(fecha_dt.strftime('%Y%m'))
    except Exception as e:
        print(f"⚠️ Error leyendo histórico: {e}")
    return 0

def listar_archivos_pendientes(session, ultima_fecha):
    """Escanea la web y devuelve URLs de archivos posteriores a la última fecha."""
    res = session.get(CONFIG["URL_WEB"])
    soup = BeautifulSoup(res.text, 'html.parser')
    
    pendientes = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        if '.rar' in href.lower():
            url = urljoin(CONFIG["BASE_URL"], href)
            nombre = url.split('/')[-1]
            
            # Regex para extraer AemetYYYY-MM
            match = re.search(r'20(\d{2})-(\d{2})', nombre)
            if match:
                fecha_int = int(f"20{match.group(1)}{match.group(2)}")
                if fecha_int >= 201401 and fecha_int > ultima_fecha:
                    pendientes.append((url, fecha_int))
    
    return sorted(pendientes, key=lambda x: x[1])

def procesar_excel(ruta_excel):
    """Limpia y extrae datos de un archivo Excel individual."""
    try:
        engine = 'xlrd' if ruta_excel.endswith('.xls') else 'openpyxl'
        df = pd.read_excel(ruta_excel, engine=engine, header=4)
        
        # Limpieza de columnas y filas
        df = df.loc[:, ~df.columns.str.contains('^Unnamed')].dropna(how='all')
        
        # Lógica de fecha multi-formato
        nombre_archivo = os.path.basename(ruta_excel)
        digitos = "".join(filter(str.isdigit, nombre_archivo))
        
        if len(digitos) == 8:
            # Intentar YYYYMMDD (Nuevo) luego DDMMYYYY (Viejo)
            fecha_obj = pd.to_datetime(digitos, format='%Y%m%d', errors='coerce')
            if pd.isnull(fecha_obj) or fecha_obj.year < 2014:
                fecha_obj = pd.to_datetime(digitos, format='%d%m%Y', errors='coerce')
            
            if pd.notnull(fecha_obj):
                df['fecha'] = fecha_obj.strftime('%Y-%m-%d')
                return df
    except Exception as e:
        print(f"  ⚠️ Error en {os.path.basename(ruta_excel)}: {e}")
    return None

def ejecutar_actualizacion():
    inicializar_entorno()
    session = requests.Session()
    session.headers.update(CONFIG["HEADERS"])
    
    ultima_fecha = obtener_ultima_fecha_registrada()
    pendientes = listar_archivos_pendientes(session, ultima_fecha)
    
    if not pendientes:
        print("✅ Sistema al día. No se requiere descarga.")
        return

    print(f"🚀 Iniciando actualización incremental: {len(pendientes)} paquetes nuevos.")
    
    df_acumulado = []
    
    for url, _ in pendientes:
        nombre_rar = url.split('/')[-1]
        ruta_rar = os.path.join(CONFIG["TEMP_DIR"], nombre_rar)
        ext_dir = os.path.join(CONFIG["TEMP_DIR"], "temp_work")
        
        try:
            print(f"⬇️ Descargando: {nombre_rar}")
            resp = session.get(url)
            with open(ruta_rar, 'wb') as f:
                f.write(resp.content)
            
            os.makedirs(ext_dir, exist_ok=True)
            subprocess.run(['unrar', 'x', '-o+', ruta_rar, ext_dir], capture_output=True, check=True)
            
            for root, _, files in os.walk(ext_dir):
                for f in files:
                    if f.lower().endswith(('.xls', '.xlsx')):
                        df_res = procesar_excel(os.path.join(root, f))
                        if df_res is not None:
                            df_acumulado.append(df_res)
                            
        except Exception as e:
            print(f"💥 Fallo crítico procesando {nombre_rar}: {e}")
        finally:
            # Limpieza garantizada de archivos temporales
            if os.path.exists(ext_dir): shutil.rmtree(ext_dir)
            if os.path.exists(ruta_rar): os.remove(ruta_rar)

    if df_acumulado:
        print("💾 Consolidando datos...")
        # Cargar histórico si existe para unirlo
        df_final = pd.concat(df_acumulado, ignore_index=True)
        
        if os.path.exists(CONFIG["ARCHIVO_FINAL"]):
            df_hist = pd.read_csv(CONFIG["ARCHIVO_FINAL"], low_memory=False)
            df_final = pd.concat([df_hist, df_final], ignore_index=True)
        
        # Eliminación de duplicados por seguridad (Estación + Fecha)
        df_final.drop_duplicates(subset=['fecha', 'Estación'], keep='last', inplace=True)
        
        # Guardado eficiente
        df_final.to_csv(CONFIG["ARCHIVO_FINAL"], index=False, encoding='utf-8-sig')
        df_final["fecha"].tail(1).to_csv(CONFIG["ULTIMA_FECHA_REGISTRADA"], index=False, encoding='utf-8-sig')
        print(f"✅ Proceso completado. Archivo actualizado: {len(df_final)} registros totales.")
    else:
        print("ℹ️ No se extrajeron nuevos datos válidos.")


ejecutar_actualizacion()

📅 Último registro local: 2026-03-31
✅ Sistema al día. No se requiere descarga.


In [14]:
# Leemos solo la columna de origen para ver el desglose
df_clima = pd.read_csv("data/datos_clima_aemet/datos_clima_2014_2026.csv")
#df_clima['fecha'] = pd.to_datetime(df_clima['fecha'], format='%Y%m%d').dt.strftime('%Y-%m-%d')
print("--- Resumen por archivo ---")
df_clima.tail()


--- Resumen por archivo ---


,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm),fecha
3569575,"El Pinar, La Dehesa",Santa Cruz de Tenerife,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-31
3569576,"San Andrés, Valverde",Santa Cruz de Tenerife,17.1 (17:30),7.3 (02:50),12.2,46 (12:30),31 (13:30),0.0,0.0,0.0,0.0,0.0,2026-03-31
3569577,Valverde,Santa Cruz de Tenerife,17.4 (15:30),10.9 (03:30),14.1,41 (15:00),29 (15:00),0.0,0.0,0.0,0.0,0.0,2026-03-31
3569578,Hierro Aeropuerto,Santa Cruz de Tenerife,23.7 (17:00),17.4 (04:10),20.6,55 (03:00),43 (02:30),0.0,0.0,0.0,0.0,0.0,2026-03-31
3569579,"Frontera, Sabinosa",Santa Cruz de Tenerife,23.0 (14:40),16.8 (07:50),19.9,40 (00:50),22 (02:20),NaN,NaN,NaN,NaN,NaN,2026-03-31


### Eliminar ultimo mes

In [ ]:
df_clima = df_clima[~df_clima["fecha"].astype(str).str.startswith("2026-03")]
df_clima.tail()
df_clima.to_csv("data/datos_clima_aemet/datos_clima_2014_2026.csv", index=False, encoding='utf-8-sig')


In [49]:
df_clima["fecha"].tail(1).to_csv("data/datos_clima_aemet/ultima_fecha_registrada.csv", index=False, encoding='utf-8-sig')

# Dias Festivos

In [15]:
ultimoAño = df_clima["fecha"].tail(1).item()
ultimoAño = pd.to_numeric(str(ultimoAño).split("-")[0])

# 1. Configurar el rango de años
years = list(range(2014, (ultimoAño+1)))

# 2. Seleccionar el país (España)
es_holidays = holidays.ES(years=years)

# 3. Crear una lista de todas las fechas en ese rango
start_date = date(2014, 1, 1)
end_date = date(ultimoAño, 12, 31)
all_days = pd.date_range(start=start_date, end=end_date)

# 4. Construir el DataFrame
df_fechas = pd.DataFrame(all_days, columns=['fecha'])

# 5. Identificar festivos y nombres
df_fechas['es_festivo'] = df_fechas['fecha'].apply(lambda x: x in es_holidays)
df_fechas['fecha'] = pd.to_datetime(df_fechas['fecha']).dt.date

# 6. Nueva Columna: Tipo de Día
def clasificar_dia(row):
    if row['es_festivo']:
        return 'Festivo'
    
    if row['fecha'].weekday() >= 5:
        return 'Fin de semana'
    
    return 'Laboral'

df_fechas['tipo_dia'] = df_fechas.apply(clasificar_dia, axis=1)
df_fechas = df_fechas.drop("es_festivo", axis=1)

# 7. Asegurar que la carpeta existe y guardar
os.makedirs('data/calendario', exist_ok=True)
df_fechas.to_csv('data/calendario/calendario_festivos_2014_2026.csv', index=False, encoding='utf-8-sig')

print("¡Archivo generado con éxito!")
# Mostrar ejemplo con diferentes tipos de días
df_fechas.tail(20)

¡Archivo generado con éxito!


,fecha,tipo_dia
4728,2026-12-12,Fin de semana
4729,2026-12-13,Fin de semana
4730,2026-12-14,Laboral
4731,2026-12-15,Laboral
4732,2026-12-16,Laboral
4733,2026-12-17,Laboral
4734,2026-12-18,Laboral
4735,2026-12-19,Fin de semana
4736,2026-12-20,Fin de semana
4737,2026-12-21,Laboral


# UNIR DATOS


In [22]:
# Unir df_extra a df_grande manteniendo la estructura de df_grande
df_clima['fecha'] = df_clima['fecha'].astype(str).str.strip()
df_fechas['fecha'] = df_fechas['fecha'].astype(str).str.strip()

df_final = pd.merge(df_clima, df_fechas, on='fecha', how='left')
df_final['fecha'] = pd.to_datetime(df_final["fecha"]).dt.date
df_final.head(50)

df_final.to_csv("Datos_final.csv", index=False, encoding='utf-8-sig')

In [23]:
df_final.tail()

,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm),fecha,tipo_dia
3569575,"El Pinar, La Dehesa",Santa Cruz de Tenerife,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-31,Laboral
3569576,"San Andrés, Valverde",Santa Cruz de Tenerife,17.1 (17:30),7.3 (02:50),12.2,46 (12:30),31 (13:30),0.0,0.0,0.0,0.0,0.0,2026-03-31,Laboral
3569577,Valverde,Santa Cruz de Tenerife,17.4 (15:30),10.9 (03:30),14.1,41 (15:00),29 (15:00),0.0,0.0,0.0,0.0,0.0,2026-03-31,Laboral
3569578,Hierro Aeropuerto,Santa Cruz de Tenerife,23.7 (17:00),17.4 (04:10),20.6,55 (03:00),43 (02:30),0.0,0.0,0.0,0.0,0.0,2026-03-31,Laboral
3569579,"Frontera, Sabinosa",Santa Cruz de Tenerife,23.0 (14:40),16.8 (07:50),19.9,40 (00:50),22 (02:20),NaN,NaN,NaN,NaN,NaN,2026-03-31,Laboral
